# Fine-tune Sentiment Model

Goal:
- Fine-tune a pretrained DistilBERT model for 3-class sentiment classification
  using the `tweet_eval` sentiment dataset.
- Labels: negative, neutral, positive
- This model will later be applied to our AI news corpus for:
  1. topic-level sentiment analysis
  2. entity-level sentiment analysis
  3. sentiment-over-time analysis

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import json

DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/nextgen_nlp_final/models/ai_sentiment_model")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
!pip install -U evaluate datasets transformers accelerate scikit-learn

In [4]:
import os
import json
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

In [5]:
from pathlib import Path
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ARTIFACTS_DIR = Path("/content/artifacts")
MODEL_DIR = ARTIFACTS_DIR / "sentiment_distilbert_tweet_eval"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS_DIR)
print("Model dir:", MODEL_DIR)

Artifacts dir: /content/artifacts
Model dir: /content/artifacts/sentiment_distilbert_tweet_eval


In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [7]:
dataset = load_dataset("tweet_eval", "sentiment")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [8]:
print(dataset["train"][0])
print(dataset["validation"][0])
print(dataset["test"][0])

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}
{'text': 'Dark Souls 3 April Launch Date Confirmed With New Trailer: Embrace the darkness.', 'label': 1}
{'text': "@user @user what do these '1/2 naked pics' have to do with anything? They're not even like that.", 'label': 1}


In [9]:
for split in ["train", "validation", "test"]:
    df_tmp = pd.DataFrame(dataset[split])
    print(f"\n{split}")
    print(df_tmp["label"].value_counts().sort_index())


train
label
0     7093
1    20673
2    17849
Name: count, dtype: int64

validation
label
0    312
1    869
2    819
Name: count, dtype: int64

test
label
0    3972
1    5937
2    2375
Name: count, dtype: int64


In [10]:
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

print(id2label)

{0: 'negative', 1: 'neutral', 2: 'positive'}


In [11]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
train_texts = dataset["train"]["text"]

token_lengths = [len(tokenizer(text, truncation=False)["input_ids"]) for text in train_texts[:5000]]

print("Length stats on first 5000 train examples:")
print(pd.Series(token_lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

Length stats on first 5000 train examples:
count    5000.0000
mean       29.8042
std         8.4628
min         7.0000
50%        30.0000
90%        40.0000
95%        43.0000
99%        52.0000
max        89.0000
dtype: float64


In [13]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [14]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]

    return {
        "accuracy": acc,
        "f1_weighted": f1
    }

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    report_to="none",
    fp16=True
)

In [17]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [19]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.622903,0.628575,0.720500,0.720803
2,0.513944,0.622119,0.749000,0.746887
3,0.422159,0.652588,0.739500,0.739372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [21]:
from google.colab import drive
from pathlib import Path
import os

trainer.save_model(str(DRIVE_MODEL_DIR))
tokenizer.save_pretrained(str(DRIVE_MODEL_DIR))

print("Model saved to:", DRIVE_MODEL_DIR)
print("Files:")
for f in os.listdir(DRIVE_MODEL_DIR):
    print("-", f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/nextgen_nlp_final/models/ai_sentiment_model
Files:
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json
- training_args.bin
